# Lab - BUILD: ShopAssist Tools Behind an MCP Server

Three tasks that move ShopAssist's tools into an MCP server. Tools that live inside one script serve only that script; behind a server that speaks the Model Context Protocol, the same tools serve any client.

| Task | What you learn |
| --- | --- |
| 1 | A tool from a typed function: name, type hints and docstring become the schema |
| 2 | The client calls, the server runs - structured errors and a call log included |
| 3 | Run it like `claude -p --allowedTools`: allow exactly the tools the task needs |

**You do not need an API key, the mcp package or Claude Code.** This lab ships with a Claude simulator and a stand-in for `mcp.server.MCPServer` when the package is not installed; `mcp_list_tools()`, `mcp_call()` and `claude_code_print()` play the client. The server code is exactly the code you would register with `claude mcp add` at home.

**You do not write code from scratch.** Each cell already holds the code, with blanks marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check
# The lab plays the MCP client: list the tools, call one, or run a `claude -p` session.
from shopassist_lab import mcp_list_tools, mcp_call, claude_code_print

# Everything below this line is ordinary MCP server code.
from mcp.server import MCPServer

ORDERS = {
    "ORD-12345678": {"item": "jacket", "status": "delivered", "delivered_days_ago": 20},
    "ORD-87654321": {"item": "lamp", "status": "delivered", "delivered_days_ago": 45},
}
CALLS = []   # the server's tiny log: one line per tool call


def log(tool, order_id):
    CALLS.append({"tool": tool, "order_id": order_id})

---

## Task 1 - A tool from a typed function

In [ ]:
# ============================================================
# TASK 1 - A tool from a typed function
# ============================================================
#
# WHAT TO DO
#   Create the server and write lookup_order as an MCP tool: a
#   Python function with a decorator. Then ask the server for
#   its tools, the way a client does.
#
# WHY IT MATTERS
#   Tools inside one script serve only that script. Behind an
#   MCP server, the same tools serve any client - Claude Code,
#   the Claude app, your own agent. In this library a tool is a
#   function: the name becomes the tool name, the type hints
#   describe the input, the docstring becomes the description
#   Claude reads. The schema is generated - not written by hand.
#
# WHERE TO SEE IT IN THE LECTURE
#   "In this library, a tool is a Python function with a
#   decorator".
#
# HOW TO DO IT
#   Three blanks:
#
#       "shopassist"   the server's name
#       str            the type of order_id - it becomes the schema
#       order          the record, when the order exists
# ============================================================

try:
    # TODO: replace each ... below with the value named beside it
    server = MCPServer(...)   # "shopassist"
    
    @server.tool()
    def lookup_order(order_id: ...) -> dict:   # str
        """Look up an order by its ID. Returns the item, the status
        and how many days ago it was delivered."""
        log("lookup_order", order_id)
        order = ORDERS.get(order_id)
        if order is None:
            return {"found": False, "message": "No order with this ID."}
        return {"found": True, "order": ...}   # order
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
tools = mcp_list_tools(server)
for t in tools:
    print(t["name"], "-", t["input_schema"])
    print("   ", t["description"].replace("\n", " "))

check("mcp_tool_schema", tools=tools)

---

## Task 2 - The client calls, the server runs

In [ ]:
# ============================================================
# TASK 2 - The client calls, the server runs
# ============================================================
#
# WHAT TO DO
#   Add start_return to the server, keeping the structured error
#   for orders past the window. Then call both tools through the
#   server, the way a client does when Claude decides to use one.
#
# WHY IT MATTERS
#   The client asks for the list of tools; when Claude chooses
#   one, the client sends the call to the server, the server
#   runs the function, and the result goes back. Your error
#   contract travels with it: the structured error reaches the
#   client unchanged. The log line per call is how you watch the
#   server work.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Start return works the same way".
#
# HOW TO DO IT
#   Three blanks:
#
#       "business"       the policy refused the action
#       "start_return"   the tool the client calls
#       "ORD-87654321"   the lamp, delivered 45 days ago
# ============================================================

try:
    # TODO: replace each ... below with the value named beside it
    @server.tool()
    def start_return(order_id: str) -> dict:
        """Start a return for a delivered order inside the 30-day
        window. Returns the return label number."""
        log("start_return", order_id)
        order = ORDERS.get(order_id)
        if order["delivered_days_ago"] > 30:
            return {"isError": True, "errorCategory": ..., "isRetryable": False,   # "business"
                    "customerMessage": "This order is past the 30-day return window.",
                    "developerMessage": f"delivered_days_ago={order['delivered_days_ago']} > 30"}
        return {"return_label": "RET-2048", "order_id": order_id}
    
    found = mcp_call(server, "lookup_order", {"order_id": "ORD-87654321"})
    refused = mcp_call(server, ..., {"order_id": ...})   # "start_return", "ORD-87654321"
except (TypeError, KeyError, ValueError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
print(found)
print(refused)
print("server log:", CALLS)

check("mcp_call", found=found, refused=refused, calls=CALLS)

---

## Task 3 - Allow exactly the tools the task needs

In [ ]:
# ============================================================
# TASK 3 - Allow exactly the tools the task needs
# ============================================================
#
# WHAT TO DO
#   Run the server's tools the way Claude Code does in print
#   mode, with an allowlist of tools: the jacket question and
#   the lamp request with both tools allowed, then the lamp with
#   lookup_order alone.
#
# WHY IT MATTERS
#   Claude Code names MCP tools mcp__<server>__<tool>, and
#   --allowedTools decides which of them Claude may use at all.
#   A question needs one lookup; a return needs both tools, and
#   the server's structured error becomes the answer. Without
#   start_return in the list, Claude cannot start a return - it
#   is not offered. Allow exactly the tools the task needs.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Now a question in print mode, with the two tools allowed".
#
# HOW TO DO IT
#   Three blanks:
#
#       "mcp__shopassist__lookup_order"   the lookup, as Claude Code names it
#       "mcp__shopassist__start_return"   the return
#       ALLOWED[:1]                       the lookup alone
# ============================================================

server.run(transport="stdio")   # provided: in a real project, `claude mcp add` starts this

try:
    # TODO: replace each ... below with the value named beside it
    ALLOWED = [..., ...]   # "mcp__shopassist__lookup_order", "mcp__shopassist__start_return"
    runs = {
        "jacket": claude_code_print("Can I still return the jacket from order ORD-12345678?",
                                    {"shopassist": server}, ALLOWED),
        "lamp": claude_code_print("I want to return the lamp from order ORD-87654321.",
                                  {"shopassist": server}, ALLOWED),
        "lookup_only": claude_code_print("I want to return the lamp from order ORD-87654321.",
                                         {"shopassist": server}, ...),   # ALLOWED[:1]
    }
except (TypeError, AttributeError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
for name, run in runs.items():
    print(f"{name}: offered {run['tools_offered']}")
    print("   ->", run["result"])

check("allowed_tools", runs=runs)

---

## Done

Put your tools behind an MCP server with clear names, typed inputs and good descriptions; register it with the client, and allow exactly the tools the task needs. One server, one contract, every Claude client.